# Integer Unpacking benchmark results
The following interaction explores the results of the benchmarks.
The controls let you explore different machines, architectures, integer types *etc*.

### Machines
The benchmarks were run on two different machines.
All benchmarks are single-threaded.

#### MacBook Pro M3
An Apple MacBook M3 Pro with 12 cores (6 performance + 6 efficiency) and 36 GB RAM.
To minimize interference, the machine was kept plugged in (Performance profile) with no other applications open and no Internet connection — necessary precautions given that macOS offers no task affinity mechanism to prioritize benchmark processes.
The full details of the machine are available in the [sysctl output](data/sysctl-osx-arm64.txt).

#### Linux x86-64 with AVX2
A [Scaleway bare metal instance](https://www.scaleway.com/en/bare-metal/) with an AMD Ryzen 5 PRO 3600 with 6 cores, and AVX2 support.
The benchmarks were run with frequency scaling disabled (`cpupower frequency-set -g performance`), pinned to a single core (`taskset`), and with no other significant process running.
The full details of the machine are available in [cpuinfo output](data/cpuinfo-linux-64.txt).

### Functions:
The various implementations being benchmarked are combinations of specific functions (source code) and microarchitectures for which they were compiled.
For instance, `Neon` on `Arm64` and `SSE4.2`, `AVX2` and `AVX512` on `x86-64`.
Some functions can be compiled for different microarchitectures, resulting in significant performance differences due to [compiler auto-vectorization](https://en.wikipedia.org/wiki/Automatic_vectorization).

The different functions are:
 - **ScalarExact**: The naive function that can unpack exactly the desired number of values. This is the function used to wrap other batch unpacking functions. 
 - **ScalarBatch**: A version without SIMD code that extracts values in batch (typically 32 values).
 - **Old**: The SIMD version that was previously in Apache Arrow and not fully leveraging SIMD insstructions.
 - **New**: The new SIMD implementation introduced with the work presented here.
 - **NewNoDispatch**: The same SIMD implementation, with all working types fixed to the one described.
 - **LittleIntPacker**: The work by Daniel Lemire available in the [LittleIntPacker library](https://github.com/fast-pack/LittleIntPacker) (under horizontal packing). This was added to this build for benchmarking and wrapped with the same bit width dispatch mechanism for a fair comparison. This is only available for the `SSE` family on `x86-64` and for `uint32_t`, but we added a few more unpacking types by extracting in a local buffer and looping on it with `static_cast` in the same way we do used for casting between different integer types.
 - **Dynamic**: Should be the best function for the architecture, wrapped in a dynamic dispatch to the appropriate function depending on the micro architecture detected at runtime.

*Note*: While `Neon` is always available on `Arm64`, we specifically compiled some functions without it to isolate the compiler's auto-vectorization capabilities.
This setting is intended for research purposes and is not generally applicable.

### Unpacked type:
This is the integer type that the unpack function targets.

### Packed width:
This is the number of bits used to encode individual integer values.
Larger bit widths can only be extracted to larger integer types.

In [ ]:
import pathlib

import polars as pl
import ipywidgets as widgets

import results_dashboard.preprocessing as preprocessing
import results_dashboard.dashboard as dashboard


def load_cleaned_benchmarks(
    arch: str = "linux-64", directory: pathlib.Path | str = "data"
) -> pl.DataFrame:
    raw_path = pathlib.Path(directory) / f"benchmark-{arch}.csv"
    processed_path = pathlib.Path(directory) / f"benchmark-{arch}.parquet"

    if preprocessing.is_emscripten() or processed_path.exists():
        return preprocessing.read_parquet(processed_path)

    # Variable number of rows to skip at the top
    n_comment_lines = count_header_lines(raw_path)
    lf_raw = pl.scan_csv(raw_path, skip_rows=n_comment_lines)
    df = preprocessing.clean_benchmark(lf_raw).collect()
    df.write_parquet(processed_path)
    return df


def make_raw_results_explorer(df: pl.DataFrame):
    arch_funcs = dashboard.make_arch_funcs(df)
    palette = dashboard.build_palette(arch_funcs)
    dashes = dashboard.build_dashes(arch_funcs)

    unpacked_type_wt = dashboard.make_unpacked_type_wt(df=df)
    func_wt = dashboard.make_func_wt(arch_funcs)
    packed_width_one_wt, packed_width_one_slider_wt = (
        dashboard.make_packed_width_one_pair_wt(unpacked_type_wt, df=df)
    )
    x_axis_wt = dashboard.make_x_axis_wt(df=df)
    out = widgets.Output()

    def plot_callback(*args, **kwargs):
        dashboard.raw_plot(
            df=df,
            unpacked_types=unpacked_type_wt.value,
            packed_width=packed_width_one_wt.value,
            arch_funcs=func_wt.value,
            x_axis=x_axis_wt.value,
            out=out,
            palette=palette,
            dashes=dashes,
            aspect=1.5,
            facet_kws={"sharex": False},
        )

    x_axis_wt.observe(plot_callback, names="value")
    unpacked_type_wt.observe(plot_callback, names="value")
    packed_width_one_wt.observe(plot_callback, names="value")
    func_wt.observe(plot_callback, names="value")

    controls = widgets.Tab(
        children=[
            # Functions
            widgets.VBox(
                [
                    widgets.Label("Select all function to compare:"),
                    func_wt,
                ]
            ),
            # Unpacked width
            widgets.VBox(
                [
                    widgets.Label("Select the target type for unpacking:"),
                    unpacked_type_wt,
                ],
                layout=widgets.Layout(min_height="100px"),
            ),
            # Packed width
            widgets.VBox(
                [
                    widgets.Label("Packed Width (in bits):"),
                    widgets.HBox([packed_width_one_slider_wt, packed_width_one_wt]),
                ]
            ),
            x_axis_wt,
        ],
        titles=["Functions", "Unpacked Type", "Packed Width", "View"],
        layout=widgets.Layout(min_height="200px"),
    )

    return widgets.VBox([controls, out]), plot_callback


def make_speed_results_explorer(df: pl.DataFrame):
    arch_funcs = dashboard.make_arch_funcs(df)
    palette = dashboard.build_palette(arch_funcs)
    dashes = dashboard.build_dashes(arch_funcs)

    unpacked_type_wt = dashboard.make_unpacked_type_wt(df=df)
    func_wt = dashboard.make_func_wt(arch_funcs)
    out = widgets.Output()

    def plot_callback(*args, **kwargs):
        dashboard.plot_speed(
            df=df,
            unpacked_types=unpacked_type_wt.value,
            arch_funcs=func_wt.value,
            out=out,
            palette=palette,
            dashes=dashes,
            aspect=1.5,
            facet_kws={"sharex": False},
        )

    unpacked_type_wt.observe(plot_callback, names="value")
    func_wt.observe(plot_callback, names="value")

    controls = widgets.Tab(
        children=[
            # Functions
            widgets.VBox(
                [
                    widgets.Label("Select all function to compare:"),
                    func_wt,
                ]
            ),
            # Unpacked width
            widgets.VBox(
                [
                    widgets.Label("Select the target type for unpacking:"),
                    unpacked_type_wt,
                ],
                layout=widgets.Layout(min_height="100px"),
            ),
        ],
        titles=["Functions", "Unpacked Type"],
        layout=widgets.Layout(min_height="200px"),
    )

    return widgets.VBox([controls, out]), plot_callback

## Full results

In [ ]:
raw_arch_wt = widgets.ToggleButtons(options=["linux-64", "osx-arm64"])
raw_controls = widgets.VBox([])


def raw_arch_callback(*args, **kwargs):
    df = load_cleaned_benchmarks(raw_arch_wt.value)
    wt, start = make_raw_results_explorer(df)
    raw_controls.children = [raw_arch_wt, wt]
    start()


raw_arch_wt.observe(raw_arch_callback, names="value")
raw_arch_callback()

raw_controls

## Speed agregation

In [ ]:
speed_arch_wt = widgets.ToggleButtons(options=["linux-64", "osx-arm64"])
speed_controls = widgets.VBox([])


def speed_arch_callback(*args, **kwargs):
    df = load_cleaned_benchmarks(speed_arch_wt.value)
    wt, start = make_speed_results_explorer(df=dashboard.linear_regression(df))
    speed_controls.children = [speed_arch_wt, wt]
    start()


speed_arch_wt.observe(speed_arch_callback, names="value")
speed_arch_callback()

speed_controls

In [ ]:
def make_blog_plot(
    arch: str, unpacked_types: list[str], arch_funcs: dict[str, list[str]]
):
    """Function used to male the image plot in the blog post."""
    import seaborn as sns

    df = dashboard.linear_regression(load_cleaned_benchmarks(arch))
    all_arch_funcs = dashboard.make_arch_funcs(df)
    palette = dashboard.build_palette(all_arch_funcs)
    dashes = dashboard.build_dashes(all_arch_funcs)

    data = (
        df.lazy()
        .filter(
            (pl.col("packed_width") != 0)
            & (pl.col("packed_width") != pl.col("unpacked_width"))
        )
        .filter(pl.col("unpacked_type").is_in(unpacked_types))
        .filter(dashboard.arch_func_filter(arch_funcs))
        .sort(by=["unpacked_type", "func", "arch"])
        .collect()
    )
    selected_funcs = [f for fs in arch_funcs.values() for f in fs]

    g = sns.relplot(
        kind="line",
        data=data,
        x="packed_width",
        y="values_per_second",
        hue="func",
        style="arch",
        col="unpacked_type",
        facet_kws={"sharex": False},
        palette=palette,
        dashes=dashes,
        col_order=unpacked_types,
        col_wrap=2,
    )

    g.figure.subplots_adjust(top=0.9, hspace=0.2)
    g.set_titles(template="Unpacking to {col_name}", y=0.95)
    g.fig.suptitle(f"Unpack speed on {arch}")
    for ax in g.axes.flat:
        scale_10 = 9
        ax.yaxis.set_major_formatter(lambda x, _: f"{x / 10**scale_10:.1f}")
        ax.set_ylabel(f"speed (×10{'⁰¹²³⁴⁵⁶⁷⁸⁹'[scale_10]} int/s)")
        ax.set_xlabel(f"Packed width (bit)")

    return g

In [ ]:
if not (out_path := pathlib.Path("static/speed-linux-64.svg")).exists():
    plot = make_blog_plot(
        arch="linux-64",
        unpacked_types=["Uint8", "Uint16", "Uint32", "Uint64"],
        arch_funcs={
            "Avx2": ["Old", "New"],
            "Sse42": ["Old", "New", "LittleIntPacker"],
            "NoSimd": ["ScalarBatch"],
        },
    )
    plot.savefig(out_path)

if not (out_path := pathlib.Path("static/speed-losx-arm64.svg")).exists():
    plot = make_blog_plot(
        arch="osx-arm64",
        unpacked_types=["Uint8", "Uint16", "Uint32", "Uint64"],
        arch_funcs={
            "Neon": ["Old", "New"],
            "NoSimd": ["ScalarBatch"],
        },
    )
    plot.savefig(out_path)